# VLM Evaluation on Training Images

This notebook evaluates the **VLM explainer used by the imaging agent** by generating explanations for a balanced sample of labeled training images.

- Source images: `dataset/archive/Training/Images`
- Metadata: `dataset/archive/stage2_train_metadata.csv`
- Label source: `Target` column (`1=positive`, `0=negative`)
- Sampling target: **10 positive + 10 negative**
- Output: generated explanations + simple explanation quality metrics

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image
import torch
import transformers
from transformers import AutoProcessor

# Same explainer model as imaging_agent.py
EXPLAINER_MODEL_ID = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"

# Resolve project root from notebook cwd
_cwd = Path.cwd()
project_root = _cwd if (_cwd / "dataset").exists() else _cwd.parent
archive_dir = project_root / "dataset" / "archive"

train_img_dir = archive_dir / "Training" / "Images"
if not train_img_dir.exists():
    raise FileNotFoundError("Could not find dataset/archive/Training/Images")

metadata_path = archive_dir / "stage2_train_metadata.csv"
print("Project root:", project_root)
print("Image dir:", train_img_dir)
print("Metadata:", metadata_path)

/Users/mathuria/Desktop/gitCode/MSDS498_Capstone_Project/.venv_mps_test311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/mathuria/Desktop/gitCode/MSDS498_Capstone_Project
Image dir: /Users/mathuria/Desktop/gitCode/MSDS498_Capstone_Project/dataset/archive/Training/Images
Metadata: /Users/mathuria/Desktop/gitCode/MSDS498_Capstone_Project/dataset/archive/stage2_train_metadata.csv


In [2]:
# Load train metadata and build patient-level labels from Target
meta_raw = pd.read_csv(metadata_path)

required_cols = {"patientId", "Target"}
missing = required_cols - set(meta_raw.columns)
if missing:
    raise ValueError(f"Missing required columns in train metadata: {missing}")

# Multiple rows can exist per patient due to multiple bounding boxes.
# Patient-level label is positive if any row has Target=1.
meta = (
    meta_raw.groupby("patientId", as_index=False)["Target"]
    .max()
    .rename(columns={"Target": "label"})
)
meta["label"] = meta["label"].astype(int)

meta["image_path"] = meta["patientId"].astype(str).map(lambda x: str(train_img_dir / f"{x}.png"))
meta["image_exists"] = meta["image_path"].map(lambda p: Path(p).exists())

print("Unique patients:", len(meta))
print("Label counts:")
print(meta["label"].value_counts(dropna=False))
print("Images found:", int(meta["image_exists"].sum()), "out of", len(meta))

Unique patients: 26684
Label counts:
label
0    20672
1     6012
Name: count, dtype: int64
Images found: 26684 out of 26684


In [3]:
# Sample 10 positive + 10 negative from TRAIN metadata
SEED = 42
SAMPLES_PER_CLASS = 10

valid = meta[(meta["image_exists"]) & (meta["label"].isin([0, 1]))].copy()
pos_pool = valid[valid["label"] == 1]
neg_pool = valid[valid["label"] == 0]

print("Positive candidates:", len(pos_pool))
print("Negative candidates:", len(neg_pool))

if len(pos_pool) < SAMPLES_PER_CLASS or len(neg_pool) < SAMPLES_PER_CLASS:
    raise ValueError(
        "Not enough labeled samples in stage2_train_metadata for 10+10 split. "
        f"Found pos={len(pos_pool)}, neg={len(neg_pool)}."
    )

pos_sample = pos_pool.sample(n=SAMPLES_PER_CLASS, random_state=SEED)
neg_sample = neg_pool.sample(n=SAMPLES_PER_CLASS, random_state=SEED)
sample_df = pd.concat([pos_sample, neg_sample], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

sample_df[["patientId", "label", "image_path"]].head(20)

Positive candidates: 6012
Negative candidates: 20672


,patientId,label,image_path
0,0c5dc7d1-7022-4e46-bc02-ff9999912edd,1,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...
1,e622bc6d-c39f-4c51-b51d-9acee5da80aa,0,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...
2,c75b22e9-76d9-4fdd-80cf-d7b88eb6d998,0,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...
3,bf842c92-c6ed-4be4-b188-c05ec8bf8dc2,1,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...
4,96f34fa9-0e56-4fdb-bfd7-781d8c453bcb,1,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...
5,5acc56a4-1b6e-4394-9364-9772047d583f,1,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...
6,83202eb5-7996-4d24-ad8b-6680d9e98bd2,0,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...
7,7f9623a6-510f-4e84-8e58-12b65a5e9dfc,1,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...
8,3ea080bf-4e81-4037-a686-fcf254a9e1bb,0,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...
9,89af4d95-4a0d-4fb3-a4e1-75d470fd3cbd,0,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...


In [4]:
# Load VLM explainer (same model family as imaging agent)
HF_TOKEN = os.getenv("HF_TOKEN")
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("Using device:", DEVICE)

def _from_pretrained_with_auth(model_cls, model_id, **kwargs):
    if HF_TOKEN:
        try:
            return model_cls.from_pretrained(model_id, token=HF_TOKEN, **kwargs)
        except TypeError:
            return model_cls.from_pretrained(model_id, use_auth_token=HF_TOKEN, **kwargs)
    return model_cls.from_pretrained(model_id, **kwargs)

def _load_explainer_model(model_id):
    preferred_classes = [
        "AutoModelForVision2Seq",
        "LlavaForConditionalGeneration",
        "LlavaNextForConditionalGeneration",
        "AutoModelForCausalLM",
    ]
    kwargs = {"torch_dtype": torch.float16} if DEVICE in {"cuda", "mps"} else {}
    last_error = None

    for class_name in preferred_classes:
        model_cls = getattr(transformers, class_name, None)
        if model_cls is None:
            continue
        try:
            print("Trying:", class_name)
            return _from_pretrained_with_auth(model_cls, model_id, **kwargs).to(DEVICE)
        except Exception as exc:
            last_error = exc
            print("  failed:", exc)

    raise RuntimeError("Unable to load explainer model") from last_error

explainer_processor = _from_pretrained_with_auth(AutoProcessor, EXPLAINER_MODEL_ID)
explainer_model = _load_explainer_model(EXPLAINER_MODEL_ID)
explainer_model.eval()
print("VLM loaded.")

Using device: mps


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Trying: LlavaForConditionalGeneration


Loading weights: 100%|██████████| 686/686 [00:19<00:00, 34.72it/s, Materializing param=model.vision_tower.vision_model.pre_layrnorm.weight]                         


VLM loaded.


In [5]:
# Prompting + generation helpers
def _build_multimodal_prompt(user_text):
    if hasattr(explainer_processor, "apply_chat_template"):
        try:
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": user_text},
                    ],
                }
            ]
            return explainer_processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=False,
            )
        except Exception:
            pass
    return f"<image>\n{user_text}"

def build_question_text(label):
    status = "positive" if int(label) == 1 else "negative"
    if status == "positive":
        return (
            f"This chest X-ray suggests {status} for pneumonia. "
            "What is the radiographic findings in this image that supports this diagnosis?"
        )
    return (
        f"This chest X-ray suggests {status} for pneumonia. "
        "What is the radiographic findings in this image that supports this diagnosis?"
    )

def generate_explanation(image_path, label, max_new_tokens=220):
    image = Image.open(image_path).convert("RGB")
    question_text = build_question_text(label)
    prompt = _build_multimodal_prompt(question_text)

    inputs = explainer_processor(images=image, text=prompt, return_tensors="pt")
    inputs = {k: v.to(DEVICE) if hasattr(v, "to") else v for k, v in inputs.items()}

    try:
        with torch.no_grad():
            generated_ids = explainer_model.generate(**inputs, max_new_tokens=max_new_tokens)
    except ValueError as exc:
        if "Image features and image tokens do not match" not in str(exc):
            raise
        fallback_prompt = f"<image>\n{question_text}"
        retry_inputs = explainer_processor(images=image, text=fallback_prompt, return_tensors="pt")
        retry_inputs = {k: v.to(DEVICE) if hasattr(v, "to") else v for k, v in retry_inputs.items()}
        with torch.no_grad():
            generated_ids = explainer_model.generate(**retry_inputs, max_new_tokens=max_new_tokens)
        inputs = retry_inputs

    input_token_count = inputs["input_ids"].shape[-1]
    generated_only = generated_ids[0][input_token_count:]
    explanation = explainer_processor.decode(generated_only, skip_special_tokens=True).strip()
    return explanation

In [6]:
# Generate explanations for sampled 20 images
rows = []
for i, row in sample_df.iterrows():
    img_path = row["image_path"]
    label = int(row["label"])
    try:
        explanation = generate_explanation(img_path, label)
        ok = True
        err = ""
    except Exception as exc:
        explanation = ""
        ok = False
        err = str(exc)

    rows.append({
        "patientId": row["patientId"],
        "label": label,
        "label_name": "positive" if label == 1 else "negative",
        "image_path": img_path,
        "success": ok,
        "error": err,
        "explanation": explanation,
        "explanation_chars": len(explanation),
        "explanation_words": len(explanation.split()) if explanation else 0,
    })
    print(f"[{i+1}/{len(sample_df)}] {row['patientId']} -> {'ok' if ok else 'failed'}")

results_df = pd.DataFrame(rows)
results_df.head()

[1/20] 0c5dc7d1-7022-4e46-bc02-ff9999912edd -> ok
[2/20] e622bc6d-c39f-4c51-b51d-9acee5da80aa -> ok
[3/20] c75b22e9-76d9-4fdd-80cf-d7b88eb6d998 -> ok
[4/20] bf842c92-c6ed-4be4-b188-c05ec8bf8dc2 -> ok
[5/20] 96f34fa9-0e56-4fdb-bfd7-781d8c453bcb -> ok
[6/20] 5acc56a4-1b6e-4394-9364-9772047d583f -> ok
[7/20] 83202eb5-7996-4d24-ad8b-6680d9e98bd2 -> ok
[8/20] 7f9623a6-510f-4e84-8e58-12b65a5e9dfc -> ok
[9/20] 3ea080bf-4e81-4037-a686-fcf254a9e1bb -> ok
[10/20] 89af4d95-4a0d-4fb3-a4e1-75d470fd3cbd -> ok
[11/20] a2cad327-ecaa-4bff-9a4c-7316e784b873 -> ok
[12/20] f92ddac9-aba6-4607-ba72-6ab7e8776fef -> ok
[13/20] b9201f87-a5ef-49f1-b45f-4c54272bc5d6 -> ok
[14/20] c50e1445-58f2-41c5-9774-2d9e24db380b -> ok
[15/20] 051e6e97-fda7-4b5c-8d5f-084bf3607d22 -> ok
[16/20] c7935568-7703-4ffe-8b55-b656cc5bb7bc -> ok
[17/20] c1f94928-371a-42d0-91ae-f959928694fe -> ok
[18/20] f8e0ed85-399a-4c57-afc5-a111923d2662 -> ok
[19/20] e701619c-8f85-4fd2-af6b-0e751d4c561d -> ok
[20/20] ad4d16e5-a851-4327-b9fe-5977da44

,patientId,label,label_name,image_path,success,error,explanation,explanation_chars,explanation_words
0,0c5dc7d1-7022-4e46-bc02-ff9999912edd,1,positive,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,True,,The radiographic findings in this chest X-ray ...,291,45
1,e622bc6d-c39f-4c51-b51d-9acee5da80aa,0,negative,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,True,,The radiographic findings in this chest X-ray ...,311,47
2,c75b22e9-76d9-4fdd-80cf-d7b88eb6d998,0,negative,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,True,,The radiographic findings in this chest X-ray ...,311,47
3,bf842c92-c6ed-4be4-b188-c05ec8bf8dc2,1,positive,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,True,,The radiographic findings in this chest X-ray ...,291,45
4,96f34fa9-0e56-4fdb-bfd7-781d8c453bcb,1,positive,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,True,,The radiographic findings in this chest X-ray ...,296,46


In [7]:
# Explanation-quality metrics (generation health + clinical keyword coverage)
POS_KEYWORDS = ["opacity", "consolidation", "infiltrate", "airspace", "lower lobe", "effusion"]
NEG_KEYWORDS = ["no focal", "clear", "no consolidation", "no effusion", "normal", "no acute"]

def keyword_hit_rate(texts, keywords):
    if len(texts) == 0:
        return np.nan
    hits = []
    for t in texts:
        tl = str(t).lower()
        hits.append(any(k in tl for k in keywords))
    return float(np.mean(hits))

ok_df = results_df[results_df["success"]].copy()
pos_ok = ok_df[ok_df["label"] == 1]
neg_ok = ok_df[ok_df["label"] == 0]

metrics = {
    "n_requested": int(len(results_df)),
    "n_success": int(results_df["success"].sum()),
    "generation_success_rate": float(results_df["success"].mean()) if len(results_df) else 0.0,
    "avg_words_all": float(ok_df["explanation_words"].mean()) if len(ok_df) else np.nan,
    "avg_words_positive": float(pos_ok["explanation_words"].mean()) if len(pos_ok) else np.nan,
    "avg_words_negative": float(neg_ok["explanation_words"].mean()) if len(neg_ok) else np.nan,
    "positive_keyword_hit_rate": keyword_hit_rate(pos_ok["explanation"].tolist(), POS_KEYWORDS),
}

pd.DataFrame({"Metric": list(metrics.keys()), "Value": list(metrics.values())})

,Metric,Value
0,n_requested,20.00
1,n_success,20.00
2,generation_success_rate,1.00
3,avg_words_all,57.15
4,avg_words_positive,54.50
5,avg_words_negative,59.80
6,positive_keyword_hit_rate,1.00


In [8]:
# Explanation-quality metrics
# - answer_relevance: semantic/lexical alignment to the task question
# - label_consistency_rate: explanation aligns with class label
# - hallucination_rate_negative: negative-label explanations that still assert positive findings
# - completeness_rate: explanation includes finding + location + confidence cues

QUESTION_TEXT = "Explain whether this chest X-ray shows evidence of pneumonia and why."


def _tokenize(text):
    return set(re.findall(r"[a-z]+", str(text).lower()))


def relevance_score(text, question):
    # Use sentence embeddings if available otherwise lexical Jaccard overla
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer("all-MiniLM-L6-v2")
        emb = model.encode([str(text), str(question)], normalize_embeddings=True)
        return float(np.dot(emb[0], emb[1]))
    except Exception:
        t1 = _tokenize(text)
        t2 = _tokenize(question)
        if not t1 or not t2:
            return np.nan
        return float(len(t1 & t2) / len(t1 | t2))


def label_consistent(text, label):
    tl = str(text).lower()
    has_pos = any(k in tl for k in POS_KEYWORDS)
    has_neg = any(k in tl for k in NEG_KEYWORDS)

    if label == 1:
        # Positive cases should mention at least one pneumonia-like finding.
        return bool(has_pos)
    # Negative cases should avoid asserting pneumonia-like findings.
    return bool(has_neg and not has_pos)


LOCATION_TERMS = [
    "left", "right", "bilateral", "upper lobe", "lower lobe", "mid lung", "base", "basilar", "perihilar"
]
CONFIDENCE_TERMS = [
    "suggest", "likely", "consistent with", "compatible with", "possible", "cannot exclude", "no evidence"
]


def has_any(text, terms):
    tl = str(text).lower()
    return any(t in tl for t in terms)


def completeness_score(text):
    # Fraction of expected explanation components present
    components = [
        has_any(text, POS_KEYWORDS + NEG_KEYWORDS),
        has_any(text, LOCATION_TERMS),
        has_any(text, CONFIDENCE_TERMS),
    ]
    return float(np.mean(components))


if len(ok_df):
    ok_df = ok_df.copy()
    ok_df["answer_relevance"] = ok_df["explanation"].map(lambda t: relevance_score(t, QUESTION_TEXT))
    ok_df["label_consistent"] = ok_df.apply(
        lambda r: label_consistent(r["explanation"], int(r["label"])), axis=1
    )
    ok_df["completeness_score"] = ok_df["explanation"].map(completeness_score)

    neg_mask = ok_df["label"] == 0
    neg_halluc = ok_df.loc[neg_mask, "explanation"].map(lambda t: has_any(t, POS_KEYWORDS))

    additional_metrics = {
        "avg_answer_relevance": float(ok_df["answer_relevance"].mean()),
        "label_consistency_rate": float(ok_df["label_consistent"].mean()),
        "hallucination_rate_negative": float(neg_halluc.mean()) if int(neg_mask.sum()) else np.nan,
        "avg_completeness_score": float(ok_df["completeness_score"].mean()),
    }
else:
    additional_metrics = {
        "avg_answer_relevance": np.nan,
        "label_consistency_rate": np.nan,
        "hallucination_rate_negative": np.nan,
        "avg_completeness_score": np.nan,
    }

pd.DataFrame({"Metric": list(additional_metrics.keys()), "Value": list(additional_metrics.values())})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1353.41it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1401.36it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1264.86it/s, Materializing param

,Metric,Value
0,avg_answer_relevance,0.774275
1,label_consistency_rate,0.500000
2,hallucination_rate_negative,1.000000
3,avg_completeness_score,0.583333


In [9]:
# Sample explanations
show_cols = ["patientId", "label_name", "success", "explanation_words", "explanation"]
display(results_df[show_cols].head(10))

out_csv = project_root / "model_exploration" / "vlm_train_explanations_results.csv"
results_df.to_csv(out_csv, index=False)

,patientId,label_name,success,explanation_words,explanation
0,0c5dc7d1-7022-4e46-bc02-ff9999912edd,positive,True,45,The radiographic findings in this chest X-ray ...
1,e622bc6d-c39f-4c51-b51d-9acee5da80aa,negative,True,47,The radiographic findings in this chest X-ray ...
2,c75b22e9-76d9-4fdd-80cf-d7b88eb6d998,negative,True,47,The radiographic findings in this chest X-ray ...
3,bf842c92-c6ed-4be4-b188-c05ec8bf8dc2,positive,True,45,The radiographic findings in this chest X-ray ...
4,96f34fa9-0e56-4fdb-bfd7-781d8c453bcb,positive,True,46,The radiographic findings in this chest X-ray ...
5,5acc56a4-1b6e-4394-9364-9772047d583f,positive,True,45,The radiographic findings in this chest X-ray ...
6,83202eb5-7996-4d24-ad8b-6680d9e98bd2,negative,True,47,The radiographic findings in this chest X-ray ...
7,7f9623a6-510f-4e84-8e58-12b65a5e9dfc,positive,True,46,The radiographic findings in this chest X-ray ...
8,3ea080bf-4e81-4037-a686-fcf254a9e1bb,negative,True,46,The radiographic findings in this chest X-ray ...
9,89af4d95-4a0d-4fb3-a4e1-75d470fd3cbd,negative,True,77,The radiographic findings in this chest X-ray ...
